# 🏗️ Facade Analyzer — Сервер для анализа фасадов

Этот notebook запускает FastAPI сервер для ML-анализа фасадов зданий.

**Требования:**
- Google Colab с GPU (T4 или лучше)
- ngrok account (бесплатный) — [регистрация](https://dashboard.ngrok.com/signup)

**Шаги:**
1. Запустите ячейку 1 — установка зависимостей
2. Запустите ячейку 2 — введите ngrok authtoken
3. Запустите ячейку 3 — запуск сервера
4. Скопируйте ngrok URL в настройки Flutter-приложения

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  ЯЧЕЙКА 1: УСТАНОВКА ЗАВИСИМОСТЕЙ               ║
# ╚══════════════════════════════════════════════════╝

!pip install fastapi uvicorn[standard] python-multipart pyngrok -q
!pip install transformers accelerate -q
!pip install ultralytics modelscope "rembg[gpu]<=2.0.50" "pillow<12.0" -q
!pip install git+https://github.com/facebookresearch/sam2.git -q
!pip install opencv-python-headless -q

# Download SAM2 weights
import os
if not os.path.exists('sam2_hiera_small.pt'):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_small.pt
    print('✅ SAM2 weights downloaded')

print('✅ Все зависимости установлены!')

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  ЯЧЕЙКА 2: НАСТРОЙКА NGROK                       ║
# ╚══════════════════════════════════════════════════╝

# Получите ваш authtoken на https://dashboard.ngrok.com/get-started/your-authtoken
NGROK_AUTHTOKEN = ''  # <-- Вставьте ваш токен сюда

if not NGROK_AUTHTOKEN:
    from getpass import getpass
    NGROK_AUTHTOKEN = getpass('Введите ngrok authtoken: ')

from pyngrok import ngrok
ngrok.set_auth_token(NGROK_AUTHTOKEN)
print('✅ ngrok настроен!')

In [ ]:
# ╔══════════════════════════════════════════════════╗
# ║  ЯЧЕЙКА 3: ЗАПУСК СЕРВЕРА                        ║
# ╚══════════════════════════════════════════════════╝

import os
import sys
import threading
import time
import logging
from contextlib import asynccontextmanager

import torch
import uvicorn
from fastapi import FastAPI, UploadFile, File, HTTPException
from fastapi.responses import FileResponse, JSONResponse
from fastapi.middleware.cors import CORSMiddleware
from pyngrok import ngrok

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# ── Import ML Pipeline ──
if not os.path.exists('ml_pipeline.py'):
    print('⚠️  ml_pipeline.py не найден!')
    print('Загрузите файлы ml_pipeline.py и repair_calculator.py в Colab.')
    print('Для этого: Files (боковая панель) → Upload to session storage')
    raise FileNotFoundError('Загрузите ml_pipeline.py и repair_calculator.py')

from ml_pipeline import FacadeAnalyzer
from repair_calculator import RepairCalculator

# ── Global State ──
analyzer = None
calculator = RepairCalculator(total_area_m2=450.0)
results_store = {}


@asynccontextmanager
async def lifespan(app):
    global analyzer
    logger.info('Loading ML models...')
    analyzer = FacadeAnalyzer()
    analyzer.load_models()
    logger.info('✅ Models loaded!')
    yield


app = FastAPI(title='Facade Analyzer', lifespan=lifespan)
app.add_middleware(
    CORSMiddleware,
    allow_origins=['*'],
    allow_methods=['*'],
    allow_headers=['*'],
)


@app.get('/')
async def root():
    return {
        'name': 'Facade Analyzer API',
        'version': '1.0.0',
        'status': 'running',
        'docs': '/docs',
        'endpoints': {
            'health': 'GET /api/health',
            'analyze': 'POST /api/analyze (multipart/form-data, field: file)',
            'images': 'GET /api/results/{id}/images/{type}',
        }
    }


@app.get('/api/health')
async def health():
    return {
        'status': 'ok',
        'models_loaded': analyzer is not None and analyzer.models_loaded,
        'device': str(analyzer.device) if analyzer else 'none',
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
    }


@app.post('/api/analyze')
async def analyze(file: UploadFile = File(...), total_area_m2: float = 450.0):
    if not analyzer or not analyzer.models_loaded:
        raise HTTPException(503, 'Models not loaded')
    
    image_bytes = await file.read()
    if not image_bytes:
        raise HTTPException(400, 'Empty file')
    
    try:
        result = analyzer.analyze(image_bytes)
        calc = RepairCalculator(total_area_m2=total_area_m2)
        repair = calc.calculate(
            damages=result['damages'],
            total_area_px=result['total_area_px'],
            layer_analysis=result.get('layer_analysis', {}),
        )
        
        results_store[result['id']] = {'image_paths': result['image_paths']}
        
        return {
            'id': result['id'],
            'overall_score': result['overall_score'],
            'overall_condition': result['overall_condition'],
            'total_area_m2': total_area_m2,
            'damaged_area_m2': round(
                result['damaged_area_px'] / result['total_area_px'] * total_area_m2
                if result['total_area_px'] > 0 else 0, 1
            ),
            'damages': result['damages'],
            'materials': result['materials'],
            'repair_estimate': repair,
            'processed_images': result['processed_images'],
        }
    except Exception as e:
        logger.exception(f'Analysis failed: {e}')
        raise HTTPException(500, str(e))


@app.get('/api/results/{aid}/images/{itype}')
async def get_image(aid: str, itype: str):
    if aid not in results_store:
        raise HTTPException(404, 'Not found')
    paths = results_store[aid].get('image_paths', {})
    if itype not in paths:
        raise HTTPException(404, f'No {itype}')
    return FileResponse(paths[itype], media_type='image/jpeg')


# ── Start server with ngrok ──
port = 8000
public_url = ngrok.connect(port)

# Extract clean URL from NgrokTunnel object
url_str = str(public_url)
if 'https://' in url_str:
    # Extract just the https://xxx.ngrok-free.app part
    import re
    match = re.search(r'(https://[^"\s]+)', url_str)
    clean_url = match.group(1) if match else url_str
else:
    clean_url = url_str

print(f'''
╔═══════════════════════════════════════════════════════════════╗
║  🚀 СЕРВЕР ЗАПУЩЕН!                                          ║
╠═══════════════════════════════════════════════════════════════╣
║                                                               ║
║  URL: {clean_url}                                             
║                                                               ║
║  Проверка в браузере:                                         ║
║  • {clean_url}/           (инфо об API)                       
║  • {clean_url}/api/health (статус сервера)                    
║  • {clean_url}/docs       (Swagger UI — тест API)             
║                                                               ║
║  Скопируйте URL в настройки Flutter-приложения                ║
║  Настройки → URL сервера → вставить URL                       ║
╚═══════════════════════════════════════════════════════════════╝
''')

# Run in thread so notebook doesn't block
def run():
    uvicorn.run(app, host='0.0.0.0', port=port, log_level='info')

thread = threading.Thread(target=run, daemon=True)
thread.start()

# Keep notebook alive
try:
    while True:
        time.sleep(1)
except KeyboardInterrupt:
    print('Сервер остановлен')
    ngrok.kill()